# Chạy CDNMF trên Art Institute of Chicago

Notebook này chạy model trực tiếp trên `Database/artic_25000`; không crawl hay tiền xử lý lại. Dữ liệu hiện có 2.149 tác phẩm, 128 feature và 8 lớp `artwork_type_title`. Chạy lần lượt các cell từ trên xuống. Mặc định chạy 1 seed, 100 epoch để bắt đầu; có thể tăng `RUNS`/`EPOCHS` sau khi xác nhận chạy thành công.

Trong Kaggle, bật GPU ở **Settings → Accelerator** nếu muốn. Kết quả được lưu trong `Database/artic_25000/model_results/`.

In [ ]:
from pathlib import Path
import json
import os
import sys

ROOT_CANDIDATES = (Path.cwd(), Path("/kaggle/working/CDNMF"))
ROOT = next(
    (
        candidate.resolve()
        for candidate in ROOT_CANDIDATES
        if (candidate / "Database" / "artic_25000" / "features.txt").is_file()
    ),
    None,
)
if ROOT is None:
    raise FileNotFoundError(
        "Không tìm thấy Database/artic_25000/features.txt. "
        "Đặt notebook trong repository CDNMF hoặc kiểm tra /kaggle/working/CDNMF."
    )

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import csv
import pickle
import random
import tempfile
import time
from copy import deepcopy

import numpy as np
import torch

from Dataset.dataset import Dataset
from Model.my_model import Model
from PreTrainer.pretrainer import PreTrainer
from Utils.evaluate import clusterscores

DATASET_DIR = ROOT / "Database" / "artic_25000"
OUTPUT_DIR = DATASET_DIR / "model_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Repository: {ROOT}")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
with (DATASET_DIR / "metadata.json").open(encoding="utf-8") as source:
    metadata = json.load(source)

required_files = ("edges.txt", "features.txt", "group.txt", "walks.txt", "nodes.csv")
missing_files = [name for name in required_files if not (DATASET_DIR / name).is_file()]
if missing_files:
    raise FileNotFoundError(f"Thiếu file dữ liệu trong {DATASET_DIR}: {missing_files}")
if metadata.get("label_field") != "artwork_type_title":
    raise ValueError("Dataset không dùng artwork_type_title làm nhãn.")

RUNS = 1
EPOCHS = 100
PRETRAIN_ITERATIONS = 100
SEED = 42
REBUILD_PRETRAIN = False

settings = {
    "net_shape": [256, 64, int(metadata["classes"])],
    "att_shape": [256, 64, int(metadata["classes"])],
    "tau": 1.3,
    "conc": 5.0,
    "negc": 400.0,
    "r": 1.0,
    "walk_weight": 1.0,
    "learning_rate": 0.001,
    "weight_decay": 0.00001,
}
PRETRAIN_PATH = ROOT / "Log" / "artic_25000" / "pretrain_params.pkl"
PRETRAIN_PATH.parent.mkdir(parents=True, exist_ok=True)

graph = Dataset({
    "feature_file": str(DATASET_DIR / "features.txt"),
    "graph_file": str(DATASET_DIR / "edges.txt"),
    "walks_file": str(DATASET_DIR / "walks.txt"),
    "label_file": str(DATASET_DIR / "group.txt"),
    "device": device,
})
if graph.num_nodes != int(metadata["processed_artworks"]):
    raise ValueError(
        f"Số node trong file ({graph.num_nodes}) không khớp metadata "
        f"({metadata['processed_artworks']})."
    )
if graph.num_classes != int(metadata["classes"]):
    raise ValueError(
        f"Số lớp trong group.txt ({graph.num_classes}) không khớp metadata "
        f"({metadata['classes']})."
    )
if graph.num_nodes != graph.X.shape[0] or graph.W.count_nonzero().item() == 0:
    raise ValueError("Feature hoặc walks.txt không khớp node hoặc không có cạnh.")
print(
    f"Đã nạp {graph.num_nodes:,} node, {int(graph.num_edges):,} cạnh, "
    f"{graph.num_feas} feature, {graph.num_classes} lớp; "
    f"{int(graph.W.count_nonzero().item()) // 2:,} cạnh walk."
)
print(f"K (số cộng đồng) = {graph.num_classes}")

In [ ]:
if REBUILD_PRETRAIN or not PRETRAIN_PATH.is_file():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    pretrainer = PreTrainer({
        "net_shape": settings["net_shape"],
        "att_shape": settings["att_shape"],
        "seed": SEED,
        "pre_iterations": PRETRAIN_ITERATIONS,
        "pretrain_params_path": str(PRETRAIN_PATH),
    })
    print("Đang pretrain từ graph và feature; bước này có thể mất vài phút.")
    pretrainer.pre_training(graph.A.detach().cpu().numpy(), "net")
    pretrainer.pre_training(graph.X.t().detach().cpu().numpy(), "att")
    print(f"Đã lưu pretrain: {PRETRAIN_PATH}")
else:
    print(f"Dùng pretrain hiện có: {PRETRAIN_PATH}")

with PRETRAIN_PATH.open("rb") as source:
    base_U, base_V = pickle.load(source)
expected_keys = {
    *(f"net{index}" for index in range(len(settings["net_shape"]))),
    *(f"att{index}" for index in range(len(settings["att_shape"]))),
}
if set(base_U) != expected_keys or set(base_V) != expected_keys:
    raise ValueError(
        "Pretrain không khớp kiến trúc hiện tại. Đặt REBUILD_PRETRAIN = True "
        "rồi chạy lại cell này."
    )
print("Pretrain đã sẵn sàng.")

In [ ]:
model_config = {
    "device": device,
    "net_shape": settings["net_shape"],
    "att_shape": settings["att_shape"],
    "net_input_dim": graph.num_nodes,
    "att_input_dim": graph.num_feas,
    "is_init": True,
    "pretrain_params_path": str(PRETRAIN_PATH),
    "tau": settings["tau"],
    "conc": settings["conc"],
    "negc": settings["negc"],
    "rec": 1.0,
    "r": settings["r"],
    "walk_weight": settings["walk_weight"],
    "learning_rate": settings["learning_rate"],
    "weight_decay": settings["weight_decay"],
    "epoch": EPOCHS,
    "run": RUNS,
    "model_path": str(OUTPUT_DIR / "artic_cdnmf_model.pt"),
}

with (DATASET_DIR / "nodes.csv").open(encoding="utf-8-sig", newline="") as source:
    nodes = list(csv.DictReader(source))
if len(nodes) != graph.num_nodes:
    raise ValueError("Số dòng nodes.csv không khớp số node của graph.")
if [int(row["node_index"]) for row in nodes] != list(range(graph.num_nodes)):
    raise ValueError("node_index trong nodes.csv không liên tục hoặc sai thứ tự.")

run_metrics = []
for run_index in range(RUNS):
    seed = SEED + run_index
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    rng = np.random.default_rng(seed)
    run_U, run_V = deepcopy(base_U), deepcopy(base_V)
    for parameters in (run_U, run_V):
        for name, values in parameters.items():
            noise = rng.normal(0, 0.05, values.shape).astype(np.float32)
            parameters[name] = np.maximum(values * np.exp(noise), 1e-8)

    with tempfile.TemporaryDirectory(prefix="artic_cdnmf_pretrain_") as temp_dir:
        run_pretrain_path = Path(temp_dir) / "pretrain_params.pkl"
        with run_pretrain_path.open("wb") as output:
            pickle.dump([run_U, run_V], output, protocol=pickle.HIGHEST_PROTOCOL)
        run_config = {**model_config, "pretrain_params_path": str(run_pretrain_path)}
        model = Model(run_config).to(device)
        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=settings["learning_rate"],
            weight_decay=settings["weight_decay"],
        )
        start_time = time.perf_counter()
        for epoch in range(1, EPOCHS + 1):
            optimizer.zero_grad(set_to_none=True)
            model()
            loss, *_ = model.loss(graph)
            if not torch.isfinite(loss).item():
                raise FloatingPointError(
                    f"Loss không hữu hạn ở run {run_index + 1}, epoch {epoch}."
                )
            loss.backward()
            optimizer.step()
            if epoch == 1 or epoch % 10 == 0 or epoch == EPOCHS:
                print(
                    f"Run {run_index + 1}/{RUNS} | epoch {epoch}/{EPOCHS} "
                    f"| loss={loss.item():.4f}"
                )

        elapsed = time.perf_counter() - start_time
        model.eval()
        with torch.no_grad():
            embeddings = model().detach().cpu().numpy()
        predicted = np.argmax(embeddings, axis=0)
        actual = graph.L.detach().cpu().numpy()
        scores = clusterscores(predicted, actual)
        run_metrics.append({
            "run": run_index + 1,
            "seed": seed,
            "epochs": EPOCHS,
            "final_loss": float(loss.detach().cpu()),
            "ACC": scores["ACC"],
            "NMI": scores["NMI"],
            "ARI": scores["ARI"],
            "AMI": scores["AMI"],
            "predicted_communities": int(np.unique(predicted).size),
            "runtime_seconds": elapsed,
        })

        predictions_path = OUTPUT_DIR / f"artic_cdnmf_assignments_seed_{seed}.csv"
        with predictions_path.open("w", encoding="utf-8-sig", newline="") as output:
            writer = csv.DictWriter(
                output,
                fieldnames=[
                    "node_index", "artwork_id", "artist_id", "title",
                    "artist_title", "label_id", "artwork_type_title",
                    "predicted_community",
                ],
            )
            writer.writeheader()
            for row, cluster in zip(nodes, predicted):
                writer.writerow({
                    **{key: row.get(key, "") for key in writer.fieldnames[:-1]},
                    "predicted_community": int(cluster),
                })

        model_path = OUTPUT_DIR / f"artic_cdnmf_model_seed_{seed}.pt"
        torch.save(
            {
                "model_state_dict": {
                    key: value.detach().cpu()
                    for key, value in model.state_dict().items()
                },
                "model_config": {
                    key: value
                    for key, value in model_config.items()
                    if key not in {"device", "pretrain_params_path", "model_path"}
                },
                "seed": seed,
            },
            model_path,
        )
        print(
            f"Run {run_index + 1}: ACC={scores['ACC']:.4f}, "
            f"NMI={scores['NMI']:.4f}, ARI={scores['ARI']:.4f}, "
            f"AMI={scores['AMI']:.4f}, "
            f"communities={np.unique(predicted).size}, time={elapsed:.1f}s"
        )
        print(f"Model: {model_path}")
        print(f"Assignments: {predictions_path}")
        del model, optimizer
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

metrics_path = OUTPUT_DIR / "artic_cdnmf_metrics.csv"
with metrics_path.open("w", encoding="utf-8-sig", newline="") as output:
    writer = csv.DictWriter(output, fieldnames=list(run_metrics[0]))
    writer.writeheader()
    writer.writerows(run_metrics)
print(f"\nĐã lưu metrics: {metrics_path}")